# Chess Coach Models: the lab's runs on a GPU

Runs the lab's own commands on this notebook's GPU: for each model in the plan, its dataset from
open Lichess data, its runs, and its leaderboard. Choose a GPU runtime (Runtime, Change runtime
type), set the plan in the first cell (a model and its runs, the name of a plan in `colab/plans/`,
or several names joined with `+` to run them as one pass), and run every cell.

The cells, in order: the plan; the GPU; the clone and its packages; the datasets; a check of every
run on a small sample; the full runs; the leaderboards; the end of the pass. To train a second
plan on the same runtime, set the first cell, run it, and run every cell after the clone again:
the clone cell starts from an empty folder, so running it again would delete the datasets and the
runs already there.

**A pass ends by itself.** The last cell packs two files: `/content/lab-results-<plan>.zip`, the
run records, leaderboards and logs with each run's test predictions where they are small, and
`/content/lab-runs-<plan>.zip`, every run's files, its trained weights among them. It asks the
browser to save the first, which is all a report is written from, and it sets the runtime to hand
itself back to Colab 45 minutes later (`RELEASE_AFTER_MINUTES` in the first cell; 0 leaves it
running). A runtime costs the same idle as training and its disk goes with it, so **save the
second file from the Files panel before then** when the weights are wanted. A pass that a failing
run stops ends the same way, with what had finished. Unzip both files at the root of a clone of
the repository (the weights land under `artifacts/runs/`, which git ignores), then
`lab eval <model>` for each model, `lab report`, commit and push.

The notebook holds no credentials and pushes nothing. `docs/compute.md` in the repository says
what runs where.

In [ ]:
# The plan: each model to run, and its runs. "" trains every run of a model's leaderboard, and
# "study:<name>" stands for every run of one of its studies. To add runs to a model that already
# has results, name them with gbm-hand among them, as in "gbm-hand,transformer-medium-long" or
# "gbm-hand,study:games": every other run is compared with gbm-hand trained on this machine.
# PLAN can also be the name of a plan kept in colab/plans/: PLAN = "studies" reads
# colab/plans/studies.yaml once the repository is cloned, and PLAN = "sets+games+moves" reads
# three plans into one pass.
PLAN = {
    "puzzle-difficulty": "",
    "puzzle-difficulty-unsolved": "",
    "where-to-look": "",
    "puzzle-themes": "",
    "blunder-hazard": "",
    "think-time": "",
    "tactic-found": "",
    "practical-chances": "",
    "game-rating": "",
    "human-move": "",
}
REF = "main"
# When a pass has ended, the runtime hands itself back to Colab after this many minutes, with
# everything on its disk. 0 leaves it running until it is deleted by hand.
RELEASE_AFTER_MINUTES = 45

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2

In [ ]:
%cd /content
!rm -rf chess-coach-models
!git clone --quiet --depth 1 --branch {REF} https://github.com/Aneesh-Pothuru/chess-coach-models.git
%cd /content/chess-coach-models
!git log --oneline -1
# Colab ships Torch with CUDA. The lab's other needs are small, and --no-deps keeps Torch as it is.
# Colab's own Polars is too old for the tactics dataset: 1.35 cannot open its source files, which
# hold a JSON-typed column, and the build then hangs. The floor replaces it. pip warns that
# Colab's cudf-polars wants the older one; the lab does not use it.
!pip install -q python-chess zstandard lightgbm 'polars>=2.1' pyyaml
!pip install -q --no-deps -e .
# Maia-2's published model is a run of human-move. Its package pins exact versions of Torch, NumPy and
# pandas, so it is installed without them, and then the three small packages it needs beside what is
# here. Where it does not install, the runs that need it wait and every other run goes ahead.
!pip install -q --no-deps maia2==0.10.0 && pip install -q einops pyzstd gdown || echo "maia2 did not install: the runs marked needs: maia2 will wait"
!python -c "import torch, lightgbm, polars, chess, zstandard; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# Each model's dataset, built once from open data and kept on this runtime. A model that already
# has results is given new runs only on the build its committed records name: a run on another
# build could not share its leaderboard, so this stops before any GPU time is spent on one.
import json
import sys
from pathlib import Path

import yaml

# What the notebook does that is worth a test is in the lab's own code, which the clone cell installed.
sys.path.insert(0, "/content/chess-coach-models/src")
from chess_coach_models import colab

shell = get_ipython()
if colab.cancel_release():
    print("a new plan starts: the release this runtime had set is cancelled")
NAME, STEPS = colab.plan_name(PLAN), colab.read_plan(PLAN)
print(f"the plan ({NAME}):")
for model, runs in STEPS.items():
    print(f"  {model}: {runs or 'every run of its leaderboard'}")
tasks = {model: yaml.safe_load(Path(f"models/{model}/task.yaml").read_text()) for model in STEPS}
for dataset in sorted({task["dataset"] for task in tasks.values()}):
    shell.system(f"lab data {dataset}")
    if shell.user_ns.get("_exit_code", 0) != 0:
        raise colab.stop(f"the build of {dataset} failed", RELEASE_AFTER_MINUTES)
for model, runs in STEPS.items():
    name = yaml.safe_load(Path(f"datasets/{tasks[model]['dataset']}.yaml").read_text())["name"]
    built = json.loads(Path(f"data/processed/{name}/manifest.json").read_text())["fingerprint"]
    records = [
        json.loads(path.read_text())
        for path in Path(f"models/{model}/results").glob("*.json")
        if path.name not in ("leaderboard.json", "analysis.json")
    ]
    named = {record["dataset"].get("fingerprint", "unnamed") for record in records}
    print(f"{model}: {name} is build {built[:12]}; its committed runs read {sorted(one[:12] for one in named) or 'none yet'}")
    if runs and named and named != {built}:
        raise colab.stop(
            f"{model}: this build of {name} is not the one its committed runs read. "
            f'Take {model} out of PLAN, or set its runs to "" to train every run on this build.',
            RELEASE_AFTER_MINUTES,
        )

In [ ]:
# A check that every run in the plan works on this GPU: each model's runs on a small sample, a
# few minutes in all. It writes under artifacts/smoke/ and is never a result.
shell = get_ipython()
for model, runs in STEPS.items():
    command = f"lab all {model} --smoke" + (f" --runs {runs}" if runs else "")
    print(f"\n=== {command}", flush=True)
    shell.system(command)
    if shell.user_ns.get("_exit_code", 0) != 0:
        raise colab.stop(f"the smoke run of {model} failed: fix that before spending GPU time on the full runs", RELEASE_AFTER_MINUTES)

In [ ]:
import os

os.makedirs("artifacts/logs", exist_ok=True)
shell = get_ipython()
colab.cancel_release()
for model, runs in STEPS.items():
    command = f"lab all {model}" + (f" --runs {runs}" if runs else "")
    print(f"\n=== {command}", flush=True)
    shell.system(f"bash -c 'set -o pipefail; {command} 2>&1 | tee artifacts/logs/{model}.log'")
    if shell.user_ns.get("_exit_code", 0) != 0:
        # What had finished is still a result: it is packed and sent as at the end of a pass.
        colab.wrap_up(NAME, RELEASE_AFTER_MINUTES)
        raise SystemExit(f"{command} failed: read artifacts/logs/{model}.log")

In [ ]:
from pathlib import Path

for model in STEPS:
    print(Path(f"models/{model}/results/leaderboard.md").read_text())

In [ ]:
# The end of the pass. What the runs made exists nowhere but on this runtime: it is packed, the
# file a report is written from is sent to the browser that opened this notebook, and the runtime
# is set to hand itself back to Colab.
colab.wrap_up(NAME, RELEASE_AFTER_MINUTES)